In [3]:
import polars as pl
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
 
BASE_PATH = Path.cwd()
SRC_DIR   = BASE_PATH / '0_source'
OUT_DIR   = BASE_PATH / '1_des'
SRC_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)
 
# 1) Load canned context
ctx = pd.read_excel(
    r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\Text_Mining\context.xlsx"
)[['Article','References','Canned Response']].dropna()
 
articles   = ctx['Article'].tolist()
references = ctx['References'].tolist()
templates  = ctx['Canned Response'].tolist()
 
# 2) Fit TF‑IDF on templates
vectorizer = TfidfVectorizer().fit(templates)
 
# 3) Process each Parquet
for pq in SRC_DIR.glob("*.parquet"):
    print(f"→ Processing {pq.name}")
 
    # A) Load & filter Participant Type
    df_pl = pl.read_parquet(pq)
    if 'Participant Type' in df_pl.columns:
        df_pl = df_pl.filter(pl.col('Participant Type') == 'HumanAgent')
 
    # B) Convert to pandas for TF‑IDF
    df = df_pl.to_pandas()
    texts = df['Text'].fillna('').tolist()
 
    # C) Vectorize
    tfidf_texts = vectorizer.transform(texts)
    tfidf_tmpls = vectorizer.transform(templates)
 
    # D) Compute cosine similarities
    sims = cosine_similarity(tfidf_texts, tfidf_tmpls)
 
    # E) Pick best match ≥ threshold
    thresh = 0.5
    is_canned      = []
    matched_canned = []
    matched_art    = []
    matched_ref    = []
    match_score    = []
 
    for row in sims:
        idx   = row.argmax()
        score = row[idx]
        if score >= thresh:
            is_canned.append(True)
            matched_canned.append(templates[idx])
            matched_art.append(articles[idx])
            matched_ref.append(references[idx])
            match_score.append(score)
        else:
            is_canned.append(False)
            matched_canned.append(None)
            matched_art.append(None)
            matched_ref.append(None)
            match_score.append(0.0)
 
    # F) Attach back into DataFrame
    df['is_canned']          = is_canned
    df['matched_canned']     = matched_canned    # <-- Canned Response
    df['matched_article']    = matched_art
    df['matched_references'] = matched_ref
    df['match_score']        = match_score
 
    # G) Sample & print with Canned Response
    matched = df[df['is_canned']]
    print(f"  • {len(matched)} rows matched (score≥{thresh}). Sample:")
    print(
        matched[['Text','matched_canned','match_score']]
        .head(5)
        .to_string(index=False)
    )
 
    # H) Aggregate usage
    stats = (
        matched
        .groupby(['matched_article','matched_references','matched_canned'])
        .size()
        .reset_index(name='usage_count')
        .sort_values('usage_count', ascending=False)
    )
    print("\n  • Top usage:")
    print(stats.head(5).to_string(index=False))
 
    # I) Export detailed + stats
    stem = pq.stem
    df.to_csv(OUT_DIR / f"{stem}_tfidf_detailed.csv", index=False)
    stats.to_csv(OUT_DIR / f"{stem}_tfidf_stats.csv",    index=False)
    print(f"    → {stem}_tfidf_detailed.csv ({len(df)} rows)")
    print(f"    → {stem}_tfidf_stats.csv   ({len(stats)} groups)\n")

→ Processing Transcript_CNX 2025-07-24T1757.parquet
  • 82319 rows matched (score≥0.5). Sample:
                                                                                                                       Text                                                           matched_canned  match_score
Thank you for staying connected. I will need just a few more minutes to look into this for you. I appreciate your patience. Thanks for your patience, just a few more minutes while I check on this.     0.500215
                                                                                         kindly keep your chat window open.              Please keep this chat window open so we can stay connected.     0.678735
                                                                                         kindly keep your chat window open.              Please keep this chat window open so we can stay connected.     0.678735
                                                                

In [4]:
df

,,Conversation Id,Agent People Id,Agent People Id_duplicated_0,Sent Time,Participant Type,Text,Translated Text,Agent Queue Group Name,Joined Time,Left Time,Latest VA Product,Latest VA Intent,is_canned,matched_canned,matched_article,matched_references,match_score
0,3,0948ffea-6712-46f6-b58f-32cd2d0cc57f,956107691.0,400651160,2025-07-24 01:59:55,HumanAgent,Please allow me 3-5 minutes to more coordinate...,None,Chat_OD_EN_Car_Activity,2025-07-22 19:25:31,2025-07-22 19:33:27,CAR,PAYMENT/RECEIPT,False,None,None,None,0.0
1,4,0948ffea-6712-46f6-b58f-32cd2d0cc57f,956107691.0,582111704,2025-07-24 01:59:55,HumanAgent,Please allow me 3-5 minutes to more coordinate...,None,Chat_OD_EN_Car_Activity,2025-07-23 18:18:36,2025-07-23 18:31:12,CAR,PAYMENT/RECEIPT,False,None,None,None,0.0
2,5,900a6726-73b5-46b7-af9d-99edeb5a52aa,449532710.0,135708935,2025-07-24 01:59:55,HumanAgent,None,None,Chat_OD_EN_Dual_GDS,2025-07-23 22:22:15,2025-07-23 22:36:21,FLIGHT,CANCEL,False,None,None,None,0.0
3,6,0948ffea-6712-46f6-b58f-32cd2d0cc57f,956107691.0,587981954,2025-07-24 01:59:55,HumanAgent,Please allow me 3-5 minutes to more coordinate...,None,Chat_OD_EN_Car_Activity,2025-07-23 18:55:20,2025-07-23 19:15:36,CAR,PAYMENT/RECEIPT,False,None,None,None,0.0
4,7,0948ffea-6712-46f6-b58f-32cd2d0cc57f,956107691.0,587981954,2025-07-24 01:59:55,HumanAgent,Please allow me 3-5 minutes to more coordinate...,None,Chat_OD_EN_Car_Activity,2025-07-22 18:44:57,2025-07-22 19:06:08,CAR,PAYMENT/RECEIPT,False,None,None,None,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
775560,1960742,3624926f-5574-42cc-b6b9-fbb58730e958,609309868.0,536504097,2025-06-26 09:04:35,HumanAgent,"But first, Is this the Itinerary that you are ...",None,Chat_OD_EN_Car_Activity,2025-07-21 16:12:31,2025-07-21 16:15:08,LODGING,CANCEL,False,None,None,None,0.0
775561,1960744,3624926f-5574-42cc-b6b9-fbb58730e958,609309868.0,536504097,2025-06-26 09:03:42,HumanAgent,Thank you for letting me know about the issue ...,None,Chat_OD_EN_Car_Activity,2025-07-21 16:12:31,2025-07-21 16:15:08,LODGING,CANCEL,False,None,None,None,0.0
775562,1960746,3624926f-5574-42cc-b6b9-fbb58730e958,609309868.0,536504097,2025-06-26 09:02:10,HumanAgent,May I know who I am chatting with?,None,Chat_OD_EN_Car_Activity,2025-07-21 16:12:31,2025-07-21 16:15:08,LODGING,CANCEL,False,None,None,None,0.0
775563,1960747,3624926f-5574-42cc-b6b9-fbb58730e958,609309868.0,536504097,2025-06-26 09:02:08,HumanAgent,"Hi! This is Gio, your go-to support for lodgin...",None,Chat_OD_EN_Car_Activity,2025-07-21 16:12:31,2025-07-21 16:15:08,LODGING,CANCEL,False,None,None,None,0.0
